In [4]:
import torch

train_graphs = torch.load(
    "processed_data/train_graphs.pt",
    weights_only=False
)

print("Train graphs:", len(train_graphs))
print(train_graphs[0])

Train graphs: 503
Data(x=[10, 6], edge_index=[2, 45], edge_attr=[45, 5], y=[45], time=1496628000, source_day='2017-06-05')


In [5]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(
    train_graphs,
    batch_size=8,
    shuffle=True
)

print("✅ DataLoader created")

✅ DataLoader created


In [6]:
batch = next(iter(train_loader))

print(batch)

DataBatch(x=[369, 6], edge_index=[2, 9240], edge_attr=[9240, 5], y=[9240], time=[8], source_day=[8], batch=[369], ptr=[9])


In [7]:
print("Node features:", batch.x.shape)
print("Edges:", batch.edge_index.shape)
print("Edge attributes:", batch.edge_attr.shape)
print("Labels:", batch.y.shape)

Node features: torch.Size([369, 6])
Edges: torch.Size([2, 9240])
Edge attributes: torch.Size([9240, 5])
Labels: torch.Size([9240])


In [8]:
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class ConflictGCN(nn.Module):
    def __init__(self, node_dim=6, edge_dim=5, hidden_dim=64):
        super().__init__()

        self.conv1 = GCNConv(node_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + edge_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index))
        x = self.conv2(x, data.edge_index)

        src = x[data.edge_index[0]]
        dst = x[data.edge_index[1]]

        edge_input = torch.cat([src, dst, data.edge_attr], dim=1)

        return self.edge_mlp(edge_input).squeeze(-1)

model = ConflictGCN()

print(model)

ConflictGCN(
  (conv1): GCNConv(6, 64)
  (conv2): GCNConv(64, 64)
  (edge_mlp): Sequential(
    (0): Linear(in_features=133, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [9]:
batch = next(iter(train_loader))

with torch.no_grad():
    logits = model(batch)

print(logits.shape)
print(batch.y.shape)

torch.Size([10468])
torch.Size([10468])


In [10]:
with torch.no_grad():
    logits = model(batch)

print("Logits:", logits.shape)
print("Labels:", batch.y.shape)

Logits: torch.Size([10468])
Labels: torch.Size([10468])


In [11]:
device = torch.device("cpu")

model = model.to(device)
batch = batch.to(device)

criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

optimizer.zero_grad()

logits = model(batch)

loss = criterion(logits, batch.y.float())

print("Loss:", loss.item())

loss.backward()

optimizer.step()

print("✅ One training step completed successfully.")

Loss: 41.96763610839844
✅ One training step completed successfully.


In [12]:
for epoch in range(1, 3):
    loss = train_epoch(
        gcn_model,
        train_loader,
        gcn_optimizer,
        gcn_criterion,
        device
    )

    print(epoch, loss)

NameError: name 'train_epoch' is not defined